# 06. External validation — Part 3: NHANES preparation and feature mapping

## Цель

Теперь, когда `READY_FOR_EXTERNAL_INFERENCE = True`, внутренний ML pipeline считаем **замороженным**.

В этой части мы **не запускаем модели** и **не меняем их**.

Здесь мы:

1. подключаем внешний датасет NHANES August 2021–August 2023;
2. скачиваем только нужные публичные таблицы;
3. объединяем их по `SEQN`;
4. сопоставляем переменные NHANES с нашими 37 input features;
5. приводим очевидные единицы измерения;
6. считаем feature coverage;
7. определяем, какие наши targets можно проверять на NHANES честно, а где есть только proxy / недостаточно данных;
8. сохраняем подготовленную external matrix для Part 4.

### Почему NHANES 2021–2023

Это актуальный публичный двухлетний цикл после возобновления полевых обследований.
В нём доступны CBC, ferritin, transferrin receptor, serum folate, hs-CRP,
standard biochemistry и demographic data.

Важно: в текущем цикле нет полного набора маркеров для всех 6 наших дефицитов,
поэтому мы **не будем выдумывать ground truth там, где его нет**.


In [1]:
from pathlib import Path
import json
import sys
import urllib.request
import warnings

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 200)

print("Python:", sys.version.split()[0])
print("Текущая рабочая папка:", Path.cwd())


Python: 3.11.16
Текущая рабочая папка: /home/sinopah/meditron/notebooks


## 1. Находим проект и проверяем frozen inference bundle


In [2]:
def find_project_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()

    for candidate in [start, *start.parents]:
        if (candidate / "data" / "raw" / "deficiency_anemia.csv").exists():
            return candidate

    raise FileNotFoundError(
        "Не найден корень проекта с data/raw/deficiency_anemia.csv"
    )


PROJECT_ROOT = find_project_root()

INTERNAL_DATA_PATH = PROJECT_ROOT / "data" / "raw" / "deficiency_anemia.csv"
BUNDLE_DIR = PROJECT_ROOT / "artifacts" / "inference_bundle"
EXTERNAL_RAW_DIR = PROJECT_ROOT / "data" / "external" / "nhanes_2021_2023" / "raw"
EXTERNAL_PROCESSED_DIR = PROJECT_ROOT / "data" / "external" / "nhanes_2021_2023" / "processed"
EXTERNAL_ARTIFACTS_DIR = PROJECT_ROOT / "artifacts" / "external_validation"

EXTERNAL_RAW_DIR.mkdir(parents=True, exist_ok=True)
EXTERNAL_PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
EXTERNAL_ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

required_bundle_files = [
    BUNDLE_DIR / "manifest.json",
    BUNDLE_DIR / "feature_contract.json",
    BUNDLE_DIR / "ensemble_meta_settings.csv",
]

missing_bundle_files = [
    str(path) for path in required_bundle_files
    if not path.exists()
]

if missing_bundle_files:
    raise FileNotFoundError(
        "Frozen inference bundle неполный:\n" +
        "\n".join(missing_bundle_files)
    )

print("PROJECT_ROOT:", PROJECT_ROOT)
print("BUNDLE_DIR:", BUNDLE_DIR)
print("Frozen bundle base files: OK")


PROJECT_ROOT: /home/sinopah/meditron
BUNDLE_DIR: /home/sinopah/meditron/artifacts/inference_bundle
Frozen bundle base files: OK


## 2. Загружаем feature contract

Это единственный допустимый список признаков для внешнего inference.


In [3]:
with open(BUNDLE_DIR / "feature_contract.json", "r", encoding="utf-8") as f:
    feature_contract = json.load(f)

FEATURE_COLUMNS = feature_contract["features"]

assert len(FEATURE_COLUMNS) == 37

print("Frozen feature count:", len(FEATURE_COLUMNS))
print(FEATURE_COLUMNS)


Frozen feature count: 37
['age_years', 'sex', 'hemoglobin', 'RBC', 'hematocrit', 'MCV', 'MCH', 'MCHC', 'RDW', 'platelets', 'WBC', 'reticulocytes', 'ferritin', 'serum_iron', 'transferrin', 'TIBC', 'UIBC', 'TSAT', 'sTfR', 'Ret_He', 'vitamin_B12', 'active_B12', 'MMA', 'homocysteine', 'folate', 'vitamin_B6', 'copper', 'ceruloplasmin', 'CRP', 'ESR', 'creatinine', 'eGFR', 'TSH', 'albumin', 'LDH', 'indirect_bilirubin', 'haptoglobin']


## 3. Файлы NHANES, которые нам нужны

Используем официальные XPT-файлы CDC/NCHS.

Таблицы:

- `DEMO_L` — age, sex;
- `CBC_L` — hemoglobin, RBC, hematocrit, MCV, MCH, MCHC, RDW, platelets, WBC;
- `FERTIN_L` — ferritin;
- `TFR_L` — soluble transferrin receptor;
- `FOLFMS_L` — serum total folate;
- `HSCRP_L` — high-sensitivity CRP;
- `BIOPRO_L` — serum iron, creatinine, albumin, LDH и др.

Сначала скачиваем только эти таблицы.


In [4]:
NHANES_BASE_URL = (
    "https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2021/DataFiles"
)

NHANES_FILES = {
    "DEMO_L": f"{NHANES_BASE_URL}/DEMO_L.xpt",
    "CBC_L": f"{NHANES_BASE_URL}/CBC_L.xpt",
    "FERTIN_L": f"{NHANES_BASE_URL}/FERTIN_L.xpt",
    "TFR_L": f"{NHANES_BASE_URL}/TFR_L.xpt",
    "FOLFMS_L": f"{NHANES_BASE_URL}/FOLFMS_L.xpt",
    "HSCRP_L": f"{NHANES_BASE_URL}/HSCRP_L.xpt",
    "BIOPRO_L": f"{NHANES_BASE_URL}/BIOPRO_L.xpt",
}

for name, url in NHANES_FILES.items():
    print(name, "->", url)


DEMO_L -> https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2021/DataFiles/DEMO_L.xpt
CBC_L -> https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2021/DataFiles/CBC_L.xpt
FERTIN_L -> https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2021/DataFiles/FERTIN_L.xpt
TFR_L -> https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2021/DataFiles/TFR_L.xpt
FOLFMS_L -> https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2021/DataFiles/FOLFMS_L.xpt
HSCRP_L -> https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2021/DataFiles/HSCRP_L.xpt
BIOPRO_L -> https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2021/DataFiles/BIOPRO_L.xpt


## 4. Скачиваем XPT-файлы

Если файл уже существует локально, повторно не скачиваем.


In [5]:
def download_if_needed(name: str, url: str) -> Path:
    output_path = EXTERNAL_RAW_DIR / f"{name}.xpt"

    if output_path.exists() and output_path.stat().st_size > 0:
        print(f"[OK] already exists: {output_path.name}")
        return output_path

    print(f"[DOWNLOAD] {name}")
    urllib.request.urlretrieve(url, output_path)

    if not output_path.exists() or output_path.stat().st_size == 0:
        raise RuntimeError(f"Не удалось скачать {name}")

    print(
        f"[OK] {output_path.name}: "
        f"{output_path.stat().st_size / 1024:.1f} KB"
    )

    return output_path


downloaded_paths = {}

for name, url in NHANES_FILES.items():
    downloaded_paths[name] = download_if_needed(name, url)


[DOWNLOAD] DEMO_L
[OK] DEMO_L.xpt: 2521.6 KB
[DOWNLOAD] CBC_L
[OK] CBC_L.xpt: 1572.1 KB
[DOWNLOAD] FERTIN_L
[OK] FERTIN_L.xpt: 81.4 KB
[DOWNLOAD] TFR_L
[OK] TFR_L.xpt: 81.4 KB
[DOWNLOAD] FOLFMS_L
[OK] FOLFMS_L.xpt: 1093.8 KB
[DOWNLOAD] HSCRP_L
[OK] HSCRP_L.xpt: 274.0 KB
[DOWNLOAD] BIOPRO_L
[OK] BIOPRO_L.xpt: 2368.7 KB


## 5. Читаем NHANES XPT


In [6]:
nhanes_tables = {}

for name, path in downloaded_paths.items():
    table = pd.read_sas(path, format="xport", encoding="utf-8")
    nhanes_tables[name] = table
    print(f"{name}: {table.shape}")

print("\nВсе таблицы загружены.")


DEMO_L: (11933, 27)
CBC_L: (8727, 23)
FERTIN_L: (2564, 4)
TFR_L: (2564, 4)
FOLFMS_L: (8727, 16)
HSCRP_L: (8727, 4)
BIOPRO_L: (7199, 42)

Все таблицы загружены.


## 6. Проверяем ключевые переменные

Если CDC когда-либо изменит структуру файла, эта ячейка остановит notebook,
вместо того чтобы молча построить неправильный mapping.


In [7]:
EXPECTED_NHANES_VARIABLES = {
    "DEMO_L": ["SEQN", "RIDAGEYR", "RIAGENDR"],
    "CBC_L": [
        "SEQN",
        "LBXHGB",
        "LBXRBCSI",
        "LBXHCT",
        "LBXMCVSI",
        "LBXMCHSI",
        "LBXMC",
        "LBXRDW",
        "LBXPLTSI",
        "LBXWBCSI",
    ],
    "FERTIN_L": ["SEQN", "LBXFER"],
    "TFR_L": ["SEQN", "LBXTFR"],
    "FOLFMS_L": ["SEQN", "LBDFOT"],
    "HSCRP_L": ["SEQN", "LBXHSCRP"],
    "BIOPRO_L": [
        "SEQN",
        "LBDSIRSI",
        "LBDSCRSI",
        "LBXSAL",
        "LBXSLDSI",
    ],
}

variable_check_rows = []

for table_name, variables in EXPECTED_NHANES_VARIABLES.items():
    table = nhanes_tables[table_name]

    for variable in variables:
        variable_check_rows.append({
            "table": table_name,
            "variable": variable,
            "exists": variable in table.columns,
        })

variable_check = pd.DataFrame(variable_check_rows)
display(variable_check)

if not variable_check["exists"].all():
    missing = variable_check.loc[
        ~variable_check["exists"],
        ["table", "variable"]
    ]
    raise ValueError(
        "Не найдены ожидаемые NHANES variables:\n"
        + missing.to_string(index=False)
    )

print("Все ключевые NHANES variables найдены.")


,table,variable,exists
0,DEMO_L,SEQN,True
1,DEMO_L,RIDAGEYR,True
2,DEMO_L,RIAGENDR,True
3,CBC_L,SEQN,True
4,CBC_L,LBXHGB,True
5,CBC_L,LBXRBCSI,True
6,CBC_L,LBXHCT,True
7,CBC_L,LBXMCVSI,True
8,CBC_L,LBXMCHSI,True
9,CBC_L,LBXMC,True


Все ключевые NHANES variables найдены.


## 7. Объединяем таблицы по `SEQN`

`SEQN` — идентификатор участника NHANES.

Начинаем с demographics и последовательно делаем `left merge`.


In [8]:
merged = nhanes_tables["DEMO_L"][
    ["SEQN", "RIDAGEYR", "RIAGENDR"]
].copy()

for table_name in [
    "CBC_L",
    "FERTIN_L",
    "TFR_L",
    "FOLFMS_L",
    "HSCRP_L",
    "BIOPRO_L",
]:
    table = nhanes_tables[table_name]

    keep_columns = EXPECTED_NHANES_VARIABLES[table_name]
    table_small = table[keep_columns].copy()

    merged = merged.merge(
        table_small,
        on="SEQN",
        how="left",
        validate="one_to_one",
    )

print("Merged NHANES shape:", merged.shape)
display(merged.head())


Merged NHANES shape: (11933, 20)


,SEQN,RIDAGEYR,RIAGENDR,LBXHGB,LBXRBCSI,LBXHCT,LBXMCVSI,LBXMCHSI,LBXMC,LBXRDW,LBXPLTSI,LBXWBCSI,LBXFER,LBXTFR,LBDFOT,LBXHSCRP,LBDSIRSI,LBDSCRSI,LBXSAL,LBXSLDSI
0,130378.0,43.0,1.0,15.7,5.19,45.3,87.4,30.2,34.6,12.9,259.0,4.7,NaN,NaN,6.23,1.78,18.0,70.72,4.3,136.0
1,130379.0,66.0,1.0,15.2,4.59,43.9,95.8,33.0,34.5,12.8,221.0,6.3,NaN,NaN,10.70,2.03,16.0,69.84,3.9,162.0
2,130380.0,44.0,2.0,13.8,4.86,40.1,82.5,28.3,34.4,13.3,235.0,5.7,13.3,4.53,11.10,5.62,24.0,56.58,3.7,103.0
3,130381.0,5.0,2.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,130382.0,2.0,1.0,12.3,4.61,36.5,79.2,26.7,33.6,14.6,342.0,11.4,24.4,NaN,31.10,NaN,NaN,NaN,NaN,NaN


## 8. Mapping NHANES → наши 37 features

Сейчас используем только **прямые и понятные соответствия**.

Мы специально НЕ подставляем:

- total bilirubin вместо indirect bilirubin;
- RBC folate вместо serum folate;
- total iron-derived показатели вместо TIBC/TSAT;
- другие лабораторные показатели как суррогаты B12/B6/copper.

Если точного аналога нет — оставляем `NaN`.


In [9]:
MAPPING = {
    "age_years": {
        "nhanes_variable": "RIDAGEYR",
        "source_file": "DEMO_L",
        "nhanes_unit": "years",
        "internal_unit": "years",
        "conversion": "identity",
    },
    "sex": {
        "nhanes_variable": "RIAGENDR",
        "source_file": "DEMO_L",
        "nhanes_unit": "1=Male, 2=Female",
        "internal_unit": "M/F",
        "conversion": "1->M, 2->F",
    },
    "hemoglobin": {
        "nhanes_variable": "LBXHGB",
        "source_file": "CBC_L",
        "nhanes_unit": "g/dL",
        "internal_unit": "g/L",
        "conversion": "x10",
    },
    "RBC": {
        "nhanes_variable": "LBXRBCSI",
        "source_file": "CBC_L",
        "nhanes_unit": "million cells/uL",
        "internal_unit": "million cells/uL",
        "conversion": "identity",
    },
    "hematocrit": {
        "nhanes_variable": "LBXHCT",
        "source_file": "CBC_L",
        "nhanes_unit": "%",
        "internal_unit": "%",
        "conversion": "identity",
    },
    "MCV": {
        "nhanes_variable": "LBXMCVSI",
        "source_file": "CBC_L",
        "nhanes_unit": "fL",
        "internal_unit": "fL",
        "conversion": "identity",
    },
    "MCH": {
        "nhanes_variable": "LBXMCHSI",
        "source_file": "CBC_L",
        "nhanes_unit": "pg",
        "internal_unit": "pg",
        "conversion": "identity",
    },
    "MCHC": {
        "nhanes_variable": "LBXMC",
        "source_file": "CBC_L",
        "nhanes_unit": "g/dL",
        "internal_unit": "g/L",
        "conversion": "x10",
    },
    "RDW": {
        "nhanes_variable": "LBXRDW",
        "source_file": "CBC_L",
        "nhanes_unit": "%",
        "internal_unit": "%",
        "conversion": "identity",
    },
    "platelets": {
        "nhanes_variable": "LBXPLTSI",
        "source_file": "CBC_L",
        "nhanes_unit": "1000 cells/uL",
        "internal_unit": "10^9/L",
        "conversion": "numerically identical",
    },
    "WBC": {
        "nhanes_variable": "LBXWBCSI",
        "source_file": "CBC_L",
        "nhanes_unit": "1000 cells/uL",
        "internal_unit": "10^9/L",
        "conversion": "numerically identical",
    },
    "ferritin": {
        "nhanes_variable": "LBXFER",
        "source_file": "FERTIN_L",
        "nhanes_unit": "ng/mL",
        "internal_unit": "ng/mL",
        "conversion": "identity",
    },
    "serum_iron": {
        "nhanes_variable": "LBDSIRSI",
        "source_file": "BIOPRO_L",
        "nhanes_unit": "umol/L",
        "internal_unit": "umol/L",
        "conversion": "identity",
    },
    "sTfR": {
        "nhanes_variable": "LBXTFR",
        "source_file": "TFR_L",
        "nhanes_unit": "mg/L",
        "internal_unit": "mg/L",
        "conversion": "identity",
    },
    "folate": {
        "nhanes_variable": "LBDFOT",
        "source_file": "FOLFMS_L",
        "nhanes_unit": "ng/mL",
        "internal_unit": "ng/mL",
        "conversion": "identity",
    },
    "CRP": {
        "nhanes_variable": "LBXHSCRP",
        "source_file": "HSCRP_L",
        "nhanes_unit": "mg/L",
        "internal_unit": "mg/L",
        "conversion": "identity",
    },
    "creatinine": {
        "nhanes_variable": "LBDSCRSI",
        "source_file": "BIOPRO_L",
        "nhanes_unit": "umol/L",
        "internal_unit": "umol/L",
        "conversion": "identity",
    },
    "albumin": {
        "nhanes_variable": "LBXSAL",
        "source_file": "BIOPRO_L",
        "nhanes_unit": "g/dL",
        "internal_unit": "g/L",
        "conversion": "x10",
    },
    "LDH": {
        "nhanes_variable": "LBXSLDSI",
        "source_file": "BIOPRO_L",
        "nhanes_unit": "U/L",
        "internal_unit": "U/L",
        "conversion": "identity",
    },
}

mapping_table = pd.DataFrame([
    {
        "internal_feature": feature,
        **details,
    }
    for feature, details in MAPPING.items()
])

display(mapping_table)

print(
    f"Прямо сопоставлено: {len(MAPPING)} / {len(FEATURE_COLUMNS)} features"
)


,internal_feature,nhanes_variable,source_file,nhanes_unit,internal_unit,conversion
0,age_years,RIDAGEYR,DEMO_L,years,years,identity
1,sex,RIAGENDR,DEMO_L,"1=Male, 2=Female",M/F,"1->M, 2->F"
2,hemoglobin,LBXHGB,CBC_L,g/dL,g/L,x10
3,RBC,LBXRBCSI,CBC_L,million cells/uL,million cells/uL,identity
4,hematocrit,LBXHCT,CBC_L,%,%,identity
5,MCV,LBXMCVSI,CBC_L,fL,fL,identity
6,MCH,LBXMCHSI,CBC_L,pg,pg,identity
7,MCHC,LBXMC,CBC_L,g/dL,g/L,x10
8,RDW,LBXRDW,CBC_L,%,%,identity
9,platelets,LBXPLTSI,CBC_L,1000 cells/uL,10^9/L,numerically identical


Прямо сопоставлено: 19 / 37 features


## 9. Строим external feature matrix

Все 37 колонок присутствуют в итоговой таблице.

Если NHANES не содержит соответствующего показателя, колонка остаётся `NaN`.

Это принципиально важно: мы не заполняем отсутствующие клинические маркеры
суррогатами только ради увеличения coverage.


In [10]:
external_X = pd.DataFrame(index=merged.index)

# ID для дальнейшего анализа
external_X["external_patient_id"] = merged["SEQN"].astype("Int64")

# Сначала создаём все frozen features как NaN
for feature in FEATURE_COLUMNS:
    external_X[feature] = np.nan

# Прямые преобразования
external_X["age_years"] = merged["RIDAGEYR"]

external_X["sex"] = merged["RIAGENDR"].map({
    1.0: "M",
    2.0: "F",
})

external_X["hemoglobin"] = merged["LBXHGB"] * 10.0
external_X["RBC"] = merged["LBXRBCSI"]
external_X["hematocrit"] = merged["LBXHCT"]
external_X["MCV"] = merged["LBXMCVSI"]
external_X["MCH"] = merged["LBXMCHSI"]
external_X["MCHC"] = merged["LBXMC"] * 10.0
external_X["RDW"] = merged["LBXRDW"]
external_X["platelets"] = merged["LBXPLTSI"]
external_X["WBC"] = merged["LBXWBCSI"]

external_X["ferritin"] = merged["LBXFER"]
external_X["serum_iron"] = merged["LBDSIRSI"]
external_X["sTfR"] = merged["LBXTFR"]
external_X["folate"] = merged["LBDFOT"]
external_X["CRP"] = merged["LBXHSCRP"]
external_X["creatinine"] = merged["LBDSCRSI"]
external_X["albumin"] = merged["LBXSAL"] * 10.0
external_X["LDH"] = merged["LBXSLDSI"]

assert set(FEATURE_COLUMNS).issubset(external_X.columns)

print("External feature matrix shape:", external_X.shape)
display(external_X.head())


External feature matrix shape: (11933, 38)


,external_patient_id,age_years,sex,hemoglobin,RBC,hematocrit,MCV,MCH,MCHC,RDW,platelets,WBC,reticulocytes,ferritin,serum_iron,transferrin,TIBC,UIBC,TSAT,sTfR,Ret_He,vitamin_B12,active_B12,MMA,homocysteine,folate,vitamin_B6,copper,ceruloplasmin,CRP,ESR,creatinine,eGFR,TSH,albumin,LDH,indirect_bilirubin,haptoglobin
0,130378,43.0,M,157.0,5.19,45.3,87.4,30.2,346.0,12.9,259.0,4.7,NaN,NaN,18.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,6.23,NaN,NaN,NaN,1.78,NaN,70.72,NaN,NaN,43.0,136.0,NaN,NaN
1,130379,66.0,M,152.0,4.59,43.9,95.8,33.0,345.0,12.8,221.0,6.3,NaN,NaN,16.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,10.70,NaN,NaN,NaN,2.03,NaN,69.84,NaN,NaN,39.0,162.0,NaN,NaN
2,130380,44.0,F,138.0,4.86,40.1,82.5,28.3,344.0,13.3,235.0,5.7,NaN,13.3,24.0,NaN,NaN,NaN,NaN,4.53,NaN,NaN,NaN,NaN,NaN,11.10,NaN,NaN,NaN,5.62,NaN,56.58,NaN,NaN,37.0,103.0,NaN,NaN
3,130381,5.0,F,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,130382,2.0,M,123.0,4.61,36.5,79.2,26.7,336.0,14.6,342.0,11.4,NaN,24.4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,31.10,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## 10. Unit sanity check against internal dataset

Это не "подгонка".

Мы просто проверяем, что после unit conversion значения имеют сопоставимый порядок величин.
Смотрим median и IQR для пересекающихся числовых признаков.


In [11]:
internal_df = pd.read_csv(INTERNAL_DATA_PATH)

mapped_numeric_features = [
    feature
    for feature in MAPPING
    if feature != "sex"
]

unit_audit_rows = []

for feature in mapped_numeric_features:
    internal_values = pd.to_numeric(
        internal_df[feature],
        errors="coerce",
    )

    external_values = pd.to_numeric(
        external_X[feature],
        errors="coerce",
    )

    unit_audit_rows.append({
        "feature": feature,
        "internal_n": int(internal_values.notna().sum()),
        "external_n": int(external_values.notna().sum()),
        "internal_median": float(internal_values.median()),
        "external_median": float(external_values.median()),
        "internal_q25": float(internal_values.quantile(0.25)),
        "internal_q75": float(internal_values.quantile(0.75)),
        "external_q25": float(external_values.quantile(0.25)),
        "external_q75": float(external_values.quantile(0.75)),
    })

unit_audit = pd.DataFrame(unit_audit_rows)

display(unit_audit)


,feature,internal_n,external_n,internal_median,external_median,internal_q25,internal_q75,external_q25,external_q75
0,age_years,840,11933,60.000,37.00,44.0000,74.0000,13.000,62.000
1,hemoglobin,840,7593,111.700,137.00,96.5750,133.7250,128.000,147.000
2,RBC,840,7593,4.000,4.66,3.3575,4.6500,4.370,4.950
3,hematocrit,840,7593,35.100,40.60,30.6000,40.7250,38.000,43.400
4,MCV,840,7593,88.400,87.90,83.2750,94.9250,84.100,91.200
5,MCH,840,7593,28.700,29.70,26.4000,31.2000,28.300,31.000
6,MCHC,840,7593,322.600,338.00,314.3000,331.1000,332.000,343.000
7,RDW,825,7593,15.500,13.50,13.9000,17.8000,13.000,14.200
8,platelets,840,7593,266.500,259.00,213.7500,324.0000,217.000,305.000
9,WBC,840,7593,5.895,6.70,4.7575,7.3600,5.500,8.100


## 11. Feature coverage

Считаем два вида coverage:

### Schema coverage
Какая доля из 37 frozen features вообще имеет прямой аналог в NHANES.

### Patient coverage
Какая доля из 37 признаков реально заполнена у каждого конкретного участника.

Это разные вещи.


In [12]:
schema_coverage = len(MAPPING) / len(FEATURE_COLUMNS)

external_feature_only = external_X[FEATURE_COLUMNS].copy()

patient_non_missing_n = external_feature_only.notna().sum(axis=1)
patient_coverage = patient_non_missing_n / len(FEATURE_COLUMNS)

coverage_summary = pd.DataFrame({
    "metric": [
        "schema_mapped_features",
        "frozen_features_total",
        "schema_coverage",
        "median_patient_non_missing_features",
        "median_patient_coverage",
        "q25_patient_coverage",
        "q75_patient_coverage",
    ],
    "value": [
        len(MAPPING),
        len(FEATURE_COLUMNS),
        schema_coverage,
        float(patient_non_missing_n.median()),
        float(patient_coverage.median()),
        float(patient_coverage.quantile(0.25)),
        float(patient_coverage.quantile(0.75)),
    ],
})

display(coverage_summary)


,metric,value
0,schema_mapped_features,19.000000
1,frozen_features_total,37.000000
2,schema_coverage,0.513514
3,median_patient_non_missing_features,16.000000
4,median_patient_coverage,0.432432
5,q25_patient_coverage,0.054054
6,q75_patient_coverage,0.459459


In [13]:
feature_coverage = pd.DataFrame({
    "feature": FEATURE_COLUMNS,
    "mapped_in_nhanes": [
        feature in MAPPING
        for feature in FEATURE_COLUMNS
    ],
    "available_n": [
        int(external_X[feature].notna().sum())
        for feature in FEATURE_COLUMNS
    ],
    "available_pct": [
        round(float(external_X[feature].notna().mean() * 100), 2)
        for feature in FEATURE_COLUMNS
    ],
})

display(feature_coverage)


,feature,mapped_in_nhanes,available_n,available_pct
0,age_years,True,11933,100.00
1,sex,True,11933,100.00
2,hemoglobin,True,7593,63.63
3,RBC,True,7593,63.63
4,hematocrit,True,7593,63.63
5,MCV,True,7593,63.63
6,MCH,True,7593,63.63
7,MCHC,True,7593,63.63
8,RDW,True,7593,63.63
9,platelets,True,7593,63.63


## 12. Какие targets можно валидировать честно

Здесь мы намеренно различаем:

- `DIRECT / STRONG` — внешний target можно определить достаточно прямо;
- `PROXY ONLY` — есть отдельные биомаркеры, но это не полноценный независимый ground truth;
- `NOT AVAILABLE` — нужных данных в выбранном NHANES cycle нет.

Важно: если label строится из того же биомаркера, который модель использует как feature,
это **не считается независимой внешней валидацией ML**.


In [14]:
target_feasibility = pd.DataFrame([
    {
        "target": "anemia",
        "status": "DIRECT / STRONG",
        "available_external_evidence": "sex + hemoglobin",
        "use_for_external_metric": True,
        "note": (
            "Можно воспроизвести детерминированное правило анемии. "
            "Это проверяет rule layer, а не качество ML binary deficiency model."
        ),
    },
    {
        "target": "iron_deficiency",
        "status": "PROXY ONLY",
        "available_external_evidence": "ferritin + serum iron + sTfR + CBC",
        "use_for_external_metric": False,
        "note": (
            "Полного независимого gold standard нет; "
            "ferritin/sTfR одновременно являются model inputs."
        ),
    },
    {
        "target": "B12_deficiency",
        "status": "NOT AVAILABLE",
        "available_external_evidence": "нет B12/MMA/active B12 в выбранном cycle",
        "use_for_external_metric": False,
        "note": "Не строим искусственный label.",
    },
    {
        "target": "folate_deficiency",
        "status": "PROXY ONLY",
        "available_external_evidence": "serum total folate",
        "use_for_external_metric": False,
        "note": (
            "Serum folate доступен, но label на его основе будет зависеть "
            "от того же feature."
        ),
    },
    {
        "target": "B6_deficiency",
        "status": "NOT AVAILABLE",
        "available_external_evidence": "нет vitamin B6 marker",
        "use_for_external_metric": False,
        "note": "Не строим искусственный label.",
    },
    {
        "target": "copper_deficiency",
        "status": "NOT AVAILABLE",
        "available_external_evidence": "нет copper/ceruloplasmin",
        "use_for_external_metric": False,
        "note": "Не строим искусственный label.",
    },
    {
        "target": "inflammation_anemia",
        "status": "PROXY ONLY",
        "available_external_evidence": "hemoglobin + hs-CRP + ferritin + serum iron",
        "use_for_external_metric": False,
        "note": (
            "Можно исследовать clinical plausibility / phenotype, "
            "но не называть это независимым ground truth."
        ),
    },
])

display(target_feasibility)


,target,status,available_external_evidence,use_for_external_metric,note
0,anemia,DIRECT / STRONG,sex + hemoglobin,True,Можно воспроизвести детерминированное правило ...
1,iron_deficiency,PROXY ONLY,ferritin + serum iron + sTfR + CBC,False,Полного независимого gold standard нет; ferrit...
2,B12_deficiency,NOT AVAILABLE,нет B12/MMA/active B12 в выбранном cycle,False,Не строим искусственный label.
3,folate_deficiency,PROXY ONLY,serum total folate,False,"Serum folate доступен, но label на его основе ..."
4,B6_deficiency,NOT AVAILABLE,нет vitamin B6 marker,False,Не строим искусственный label.
5,copper_deficiency,NOT AVAILABLE,нет copper/ceruloplasmin,False,Не строим искусственный label.
6,inflammation_anemia,PROXY ONLY,hemoglobin + hs-CRP + ferritin + serum iron,False,Можно исследовать clinical plausibility / phen...


## 13. Создаём только внешний anemia label

Это не ML target. Он нужен для проверки переносимости детерминированного anemia rule.

Используем ту же frozen логику:

- female: Hb < 120 g/L;
- male: Hb < 130 g/L.


In [15]:
external_X["external_anemia_rule"] = np.nan

female_mask = external_X["sex"].eq("F") & external_X["hemoglobin"].notna()
male_mask = external_X["sex"].eq("M") & external_X["hemoglobin"].notna()

external_X.loc[female_mask, "external_anemia_rule"] = (
    external_X.loc[female_mask, "hemoglobin"] < 120
).astype(int)

external_X.loc[male_mask, "external_anemia_rule"] = (
    external_X.loc[male_mask, "hemoglobin"] < 130
).astype(int)

print(
    "Patients with reproducible anemia status:",
    int(external_X["external_anemia_rule"].notna().sum()),
)

print(
    "External anemia prevalence:",
    round(
        external_X["external_anemia_rule"].mean(),
        4,
    ),
)


Patients with reproducible anemia status: 7593
External anemia prevalence: 0.147


## 14. Сохраняем NHANES preparation artifacts


In [16]:
external_matrix_path = (
    EXTERNAL_PROCESSED_DIR / "nhanes_external_feature_matrix.csv"
)

mapping_path = (
    EXTERNAL_ARTIFACTS_DIR / "nhanes_feature_mapping.csv"
)

coverage_path = (
    EXTERNAL_ARTIFACTS_DIR / "nhanes_feature_coverage.csv"
)

unit_audit_path = (
    EXTERNAL_ARTIFACTS_DIR / "nhanes_unit_audit.csv"
)

target_feasibility_path = (
    EXTERNAL_ARTIFACTS_DIR / "nhanes_target_feasibility.csv"
)

external_X.to_csv(external_matrix_path, index=False)
mapping_table.to_csv(mapping_path, index=False)
feature_coverage.to_csv(coverage_path, index=False)
unit_audit.to_csv(unit_audit_path, index=False)
target_feasibility.to_csv(target_feasibility_path, index=False)

print("Saved:", external_matrix_path)
print("Saved:", mapping_path)
print("Saved:", coverage_path)
print("Saved:", unit_audit_path)
print("Saved:", target_feasibility_path)


Saved: /home/sinopah/meditron/data/external/nhanes_2021_2023/processed/nhanes_external_feature_matrix.csv
Saved: /home/sinopah/meditron/artifacts/external_validation/nhanes_feature_mapping.csv
Saved: /home/sinopah/meditron/artifacts/external_validation/nhanes_feature_coverage.csv
Saved: /home/sinopah/meditron/artifacts/external_validation/nhanes_unit_audit.csv
Saved: /home/sinopah/meditron/artifacts/external_validation/nhanes_target_feasibility.csv


# Что прислать после запуска Part 3

Пришли мне:

1. `Merged NHANES shape`;
2. строку `Прямо сопоставлено: ... / 37 features`;
3. таблицу `coverage_summary`;
4. таблицу `unit_audit` целиком;
5. `Patients with reproducible anemia status` и `External anemia prevalence`;
6. если появилась ошибка скачивания / чтения XPT — сам текст ошибки.

После проверки этих результатов перейдём к **Part 4 — frozen external inference**:

- загрузим финальные L1 + CatBoost;
- получим вероятности на NHANES;
- применим calibrators;
- применим frozen ensemble weights/thresholds;
- оценим missingness/domain shift;
- отдельно разберём, какие результаты можно называть external validation, а какие только transfer/plausibility analysis.
